# Notebook D — QUBO : sélection combinatoire de kernels

De "combiner tous les kernels" à "choisir les meilleurs" — comment formuler ce problème, le résoudre, et comprendre quand le quantique devient nécessaire.

**Plan :**
1. Le problème de sélection de sous-ensemble — NP-difficile
2. Formulation QUBO — matrice Q, intuition, exemple concret
3. Résolution classique : brute force, greedy, recuit simulé
4. Le rôle du paramètre λ — sweep et calibration
5. Connexion au recuit quantique — quand M devient grand
6. Notre résultat concret sur les 12 kernels


## Section 1 — Le problème de sélection de sous-ensemble

### Formalisation

On a $M$ kernels $\{K_1, ..., K_M\}$ et un vecteur de variables binaires $\mathbf{w} \in \{0,1\}^M$.

**Objectif :** trouver le sous-ensemble $S = \{m : w_m = 1\}$ qui maximise l'AUC du SVM entraîné sur $K_S = \sum_{m \in S} K_m / |S|$.

### Complexité combinatoire

**L'espace de recherche :** $\{0,1\}^M$ contient $2^M$ sous-ensembles.

| $M$ | $2^M$ | Temps brute force (CPU, $10^8$ éval/s) |
|-----|-------|----------------------------------------|
| 10  | 1 024 | < 1 ms |
| 12  | 4 096 | < 1 ms (notre projet) |
| 20  | $\approx 10^6$ | 10 ms |
| 25  | 33 millions | 330 ms |
| 30  | $\approx 10^9$ | 10 s |
| 40  | $\approx 10^{12}$ | 3 heures |
| 50  | $\approx 10^{15}$ | 3 ans |

### Pourquoi c'est NP-difficile

Le problème de sélection de features est une **réduction du problème de couverture maximale** (Max-Coverage), qui est NP-difficile. Cela signifie qu'aucun algorithme polynomial ne peut garantir de trouver l'optimum exact pour toute instance.

**Conséquence :** Pour $M$ grand, on doit utiliser des heuristiques (greedy, recuit simulé) ou des optimiseurs spécialisés (annealers quantiques pour QUBO).

### QUBO : reformulation analytique

Au lieu de maximiser directement l'AUC (qui nécessite d'entraîner un SVM), on maximise un **proxy analytique** :
$$\max_{\mathbf{w} \in \{0,1\}^M} \underbrace{\sum_m w_m s_m}_{\text{performance}} - \lambda \underbrace{\sum_{m \neq m'} w_m w_{m'} A_{mm'}}_{\text{pénalité redondance}}$$

où $s_m$ = AUC individuelle du kernel $m$ et $A_{mm'}$ = alignement de Frobenius entre $K_m$ et $K_{m'}$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from itertools import combinations

np.random.seed(42)

# ── Construire M=10 kernels synthétiques ────────────────────────────────
def K_Z(X, alpha=1.0):
    n, d = X.shape
    K = np.ones((n,n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha*diff)**2
    return K

def K_ZZ(X, alpha=1.0):
    K = K_Z(X, alpha)
    n, d = X.shape
    for k in range(d):
        for l in range(k+1, d):
            c = X[:,k]*X[:,l]
            diff = c.reshape(-1,1) - c.reshape(1,-1)
            K *= np.cos(alpha*diff)**2
    return K

# Données synthétiques
X_demo = np.random.uniform(0, 2, (30, 4))
y_demo = (X_demo[:,0]**2 + X_demo[:,1]**2 > 2).astype(float) * 2 - 1  # cercle

kernel_lib = (
    [(K_Z, a, f'Z α={a}') for a in [0.5, 1.0, 2.0, 3.0]] +
    [(K_ZZ, a, f'ZZ α={a}') for a in [0.5, 1.0, 2.0, 3.0]] +
    [(K_Z, a, f'Z α={a} bis') for a in [1.5, 2.5]]
)

M = len(kernel_lib)
print(f"Nombre de kernels: M = {M}")
print(f"Espace de recherche: 2^{M} = {2**M} sous-ensembles")

# Calculer les matrices kernel
Kmats = []
for fn, alpha, name in kernel_lib:
    K = fn(X_demo, alpha)
    np.fill_diagonal(K, 1.0)
    Kmats.append(K)

# ── Énumérer TOUS les sous-ensembles (brute force, possible car M=10) ────
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

def eval_subset(indices, Kmats, y, cv=3):
    '''Évaluer l'AUC d'un sous-ensemble de kernels.'''
    if len(indices) == 0:
        return 0.5
    K_comb = np.mean([Kmats[i] for i in indices], axis=0)
    eigs = np.linalg.eigvalsh(K_comb)
    if eigs.min() < 0:
        K_comb += (-eigs.min()+1e-8)*np.eye(len(K_comb))
    try:
        return cross_val_score(SVC(kernel='precomputed',C=1.), K_comb, y,
                               cv=cv, scoring='roc_auc').mean()
    except:
        return 0.5

# Tester tous les sous-ensembles de taille 1, 2, 3
print("
Énumération de tous les sous-ensembles de taille 1 à 3...")
t0 = time.time()
subset_results = {}
for size in [1, 2, 3]:
    best_auc, best_subset = 0, None
    for combo in combinations(range(M), size):
        auc = eval_subset(list(combo), Kmats, y_demo)
        subset_results[combo] = auc
        if auc > best_auc:
            best_auc, best_subset = auc, combo
    names_best = [kernel_lib[i][2] for i in best_subset]
    print(f"  Taille {size}: meilleur = {names_best}, AUC = {best_auc:.4f}")
t1 = time.time()
print(f"
Temps pour taille 1-3 ({sum(1 for _ in combinations(range(M),1))+sum(1 for _ in combinations(range(M),2))+sum(1 for _ in combinations(range(M),3))} sous-ensembles): {t1-t0:.2f}s")

# ── Visualiser le paysage combinatoire pour la taille 2 ──────────────────
fig, ax = plt.subplots(figsize=(10, 4))
combos_2 = list(combinations(range(M), 2))
aucs_2 = [subset_results[c] for c in combos_2]
indices = np.argsort(aucs_2)[::-1]  # trier par AUC décroissant

ax.bar(range(len(aucs_2)), [aucs_2[i] for i in indices], color='#3498db', alpha=0.7)
# Annoter les 3 meilleurs
for rank, idx in enumerate(range(3)):
    combo_idx = indices[idx]
    names = [kernel_lib[i][2] for i in combos_2[combo_idx]]
    ax.text(idx, aucs_2[indices[idx]] + 0.005, f'{names[0]}
+{names[1]}',
            ha='center', fontsize=7, color='#c0392b', fontweight='bold')

ax.axhline(0.5, color='grey', ls=':', lw=1, label='Aléatoire')
ax.set_xlabel(f'Sous-ensembles de taille 2 (triés par AUC décroissante)')
ax.set_ylabel('AUC (3-fold cross-val)')
ax.set_title(f'Paysage combinatoire : AUC de tous les C({M},2)={len(combos_2)} paires de kernels
'
              'Le problème de sélection = trouver le pic sans tout évaluer',
             fontweight='bold')
ax.set_xticks([])
ax.legend()
plt.tight_layout()
plt.show()


## Section 2 — Formulation QUBO : dérivation pas à pas

### De l'objectif à la matrice Q

**Objectif :** maximiser qualité - λ × redondance
$$\max_{\mathbf{w} \in \{0,1\}^M} \sum_m w_m s_m - \lambda \sum_{m \neq m'} w_m w_{m'} A_{mm'}$$

**Reformulation en minimisation** (multiplication par $-1$) :
$$\min_{\mathbf{w} \in \{0,1\}^M} -\sum_m w_m s_m + \lambda \sum_{m \neq m'} w_m w_{m'} A_{mm'}$$

**Écriture matricielle** : $\min_{\mathbf{w}} \mathbf{w}^\top \mathbf{Q} \mathbf{w}$

Développons $\mathbf{w}^\top \mathbf{Q} \mathbf{w} = \sum_{m,m'} Q_{mm'} w_m w_{m'}$ :
- Termes $m = m'$ : $\sum_m Q_{mm} w_m^2 = \sum_m Q_{mm} w_m$ (car $w_m^2 = w_m$ pour $w_m \in \{0,1\}$)
- Termes $m \neq m'$ : $\sum_{m \neq m'} Q_{mm'} w_m w_{m'}$

Identification terme par terme :
$$\boxed{Q_{mm} = -s_m \quad (\text{négatif = encourage la sélection})}$$
$$\boxed{Q_{mm'} = \lambda A_{mm'} \quad (\text{positif = pénalise la co-sélection si redondants})}$$

### Exemple concret à 3 kernels

Soit $M=3$, $s_1=0.75, s_2=0.70, s_3=0.68$, $A_{12}=0.80$ (redondants), $A_{13}=0.25, A_{23}=0.30$, $\lambda=1.0$.

$$\mathbf{Q} = \begin{pmatrix} -0.75 & 0.80 & 0.25 \\ 0.80 & -0.70 & 0.30 \\ 0.25 & 0.30 & -0.68 \end{pmatrix}$$

Évaluation des 8 solutions possibles :
- $w=(1,1,1)$ : $\mathbf{w}^\top\mathbf{Q}\mathbf{w} = -0.75-0.70-0.68+2(0.80+0.25+0.30) = -0.43$
- $w=(1,0,1)$ : $-0.75-0.68+2(0.25) = -0.93$ ← **optimum** !
- $w=(1,1,0)$ : $-0.75-0.70+2(0.80) = 0.15$ (mauvais car $K_1, K_2$ redondants)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

# ── Exemple à 3 kernels ───────────────────────────────────────────────────
s = np.array([0.75, 0.70, 0.68])    # AUC individuels
A = np.array([[1.0,  0.80, 0.25],   # alignements mutuels
              [0.80, 1.0,  0.30],
              [0.25, 0.30, 1.0 ]])
lam = 1.0

# Construire la matrice Q
M_small = len(s)
Q3 = lam * A.copy()
np.fill_diagonal(Q3, -s)
print("Matrice Q (M=3, λ=1.0):")
print(Q3.round(3))

# Énumérer les 8 solutions
print("
Énumération complète des 2^3 = 8 solutions:")
print(f"{'w':15s} | {'w^T Q w':8s} | {'Σs_m':6s} | {'λ×Σ A_mm'':9s} | Interprétation")
print("-"*70)

all_solutions = list(product([0,1], repeat=M_small))
obj_vals = []
for w_tuple in all_solutions:
    w = np.array(w_tuple, dtype=float)
    obj = w @ Q3 @ w
    perf = np.dot(w, s)
    selected = [i for i, wi in enumerate(w) if wi == 1]
    redund = sum(A[i,j] for i in selected for j in selected if i!=j) * lam
    names_sel = '+'.join([f'K{i+1}(AUC={s[i]:.2f})' for i in selected]) if selected else 'Aucun'
    interp = 'OPTIMUM' if obj == min(w @ Q3 @ w for _ in [None] for w in [np.array(x,float) for x in all_solutions]) else ''
    print(f"w={list(w_tuple)} {names_sel:30s} | {obj:8.3f} | {perf:6.3f} | {redund:9.3f} | {interp}")
    obj_vals.append(obj)

best_idx = np.argmin(obj_vals)
best_w = all_solutions[best_idx]
print(f"
Solution optimale: w={best_w}, objectif={obj_vals[best_idx]:.3f}")
print(f"Kernels sélectionnés: {[i+1 for i,wi in enumerate(best_w) if wi==1]}")
print("→ K1 et K3 sont complémentaires (A₁₃=0.25), pas K1 et K2 (A₁₂=0.80)")

# ── Visualisation de la matrice Q ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
vmax = np.max(np.abs(Q3))
im = ax.imshow(Q3, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
plt.colorbar(im, ax=ax)
for i in range(M_small):
    for j in range(M_small):
        color = 'white' if abs(Q3[i,j]) > 0.4 else 'black'
        ax.text(j, i, f'{Q3[i,j]:.3f}', ha='center', va='center',
                fontsize=12, color=color, fontweight='bold')
ax.set_xticks(range(M_small)), ax.set_yticks(range(M_small))
ax.set_xticklabels([f'K{i+1}' for i in range(M_small)], fontsize=12)
ax.set_yticklabels([f'K{i+1}' for i in range(M_small)], fontsize=12)
ax.set_title('Matrice Q (M=3, λ=1.0)
Bleu = sélectionner, Rouge = pénaliser', fontweight='bold')
# Annotation
ax.text(0.5, -0.12, 'Diagonale Q_mm = -s_m < 0 → sélectionner', ha='center',
        transform=ax.transAxes, fontsize=9, color='#2980b9')
ax.text(0.5, -0.20, 'Hors-diag Q_mm' = λ·A_mm' > 0 → pénaliser si redondants', ha='center',
        transform=ax.transAxes, fontsize=9, color='#c0392b')

ax2 = axes[1]
colors_bar = ['#c0392b' if o > 0 else '#27ae60' for o in obj_vals]
bars = ax2.bar(range(8), obj_vals, color=colors_bar, alpha=0.8)
ax2.bar(best_idx, obj_vals[best_idx], color='gold', alpha=1.0, zorder=3, label='Optimum')
for i, (bar, obj, wt) in enumerate(zip(bars, obj_vals, all_solutions)):
    ax2.text(i, obj - 0.05 if obj < 0 else obj + 0.05, str(list(wt)),
             ha='center', fontsize=7, rotation=45)
ax2.axhline(0, color='grey', lw=1, ls='--')
ax2.set_xticks(range(8))
ax2.set_xticklabels([str(list(w)) for w in all_solutions], fontsize=7, rotation=45)
ax2.set_ylabel('w^T Q w (minimiser)')
ax2.set_title('Objectif QUBO pour les 8 solutions
'
              'Or = optimum global', fontweight='bold')
ax2.legend()
plt.tight_layout()
plt.show()


## Section 3 — Résolution classique : trois algorithmes comparés

### Brute force

**Complexité :** $O(2^M)$ — exact mais exponentiel.
**Faisable pour :** $M \leq 25$ environ (quelques secondes).

### Greedy (glouton)

**Algorithme :**
1. Démarrer avec $S = \emptyset$
2. À chaque étape : ajouter le kernel $m^* = \arg\min_{m \notin S} f(\mathbf{w}_{S \cup \{m\}})$ qui diminue le plus l'objectif
3. S'arrêter quand plus d'amélioration

**Complexité :** $O(M^2)$ au plus — très rapide.
**Limite :** sous-optimal (ne garantit pas l'optimum global).

### Recuit simulé (Simulated Annealing)

**Algorithme :**
1. Démarrer avec une solution aléatoire $\mathbf{w}$, température $T$
2. À chaque itération : proposer une modification (flip un bit)
3. Accepter si amélioration, ou avec probabilité $e^{-\Delta f / T}$ si dégradation
4. Diminuer $T$ progressivement (cooling schedule)

**Complexité :** $O(M \times T_{\text{iter}})$ — heuristique mais efficace.
**Avantage :** peut s'échapper des minima locaux.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.random.seed(42)

# ── Construire une instance QUBO réaliste (M=12 kernels) ────────────────
def K_Z(X, alpha=1.0):
    n, d = X.shape
    K = np.ones((n,n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha*diff)**2
    return K

def K_ZZ(X, alpha=1.0):
    K = K_Z(X, alpha)
    n, d = X.shape
    for k in range(d):
        for l in range(k+1, d):
            c = X[:,k]*X[:,l]
            diff = c.reshape(-1,1) - c.reshape(1,-1)
            K *= np.cos(alpha*diff)**2
    return K

def center_kernel(K):
    n = len(K); H = np.eye(n) - np.ones((n,n))/n; return H@K@H

def frobenius_ip(A, B): return np.sum(A*B)

# Données synthétiques + 12 kernels
rng = np.random.default_rng(42)
X5 = np.zeros((5, 4))
X5[:3] = rng.uniform(0.2, 0.8, (3,4))
X5[3:]  = rng.uniform(1.3, 2.0, (2,4))
y5 = np.array([1.,1.,1.,-1.,-1.])

KERNELS_12 = [
    ('Z α=1.0',  K_Z,   1.0, 0.698),
    ('Z α=3.0',  K_Z,   3.0, 0.712),
    ('ZZ α=1.0', K_ZZ,  1.0, 0.734),
    ('ZZ α=4.0', K_ZZ,  4.0, 0.756),
    ('ZZ α=2.5', K_ZZ,  2.5, 0.738),
    ('ZZ α=0.5', K_ZZ,  0.5, 0.720),
    ('Z α=0.5',  K_Z,   0.5, 0.695),
    ('Z α=2.0',  K_Z,   2.0, 0.705),
    ('ZZ α=1.5', K_ZZ,  1.5, 0.730),
    ('ZZ α=3.0', K_ZZ,  3.0, 0.748),
    ('Z α=1.5',  K_Z,   1.5, 0.701),
    ('ZZ α=2.0', K_ZZ,  2.0, 0.742),
]

M = len(KERNELS_12)
s = np.array([k[3] for k in KERNELS_12])

# Calculer les alignements de Frobenius
Kmats = []
for name, fn, alpha, _ in KERNELS_12:
    K = fn(X5, alpha)
    np.fill_diagonal(K, 1.0)
    Kmats.append(K)

Kmats_c = [center_kernel(K) for K in Kmats]
A_mat = np.zeros((M, M))
for m in range(M):
    for mp in range(M):
        num = frobenius_ip(Kmats_c[m], Kmats_c[mp])
        den = np.sqrt(frobenius_ip(Kmats_c[m],Kmats_c[m]) * frobenius_ip(Kmats_c[mp],Kmats_c[mp]))
        A_mat[m,mp] = num/den if den>0 else 0.

lam = 1.0
Q_mat = lam * A_mat.copy()
np.fill_diagonal(Q_mat, -s)

def qubo_objective(w, Q):
    w = np.array(w, dtype=float)
    return w @ Q @ w

# ── ALGORITHME 1 : Brute Force ────────────────────────────────────────────
print("=== Brute Force (M=12) ===")
t0 = time.time()
best_bf = None
best_obj_bf = np.inf
from itertools import product as iproduct
for bits in iproduct([0,1], repeat=M):
    w = np.array(bits, dtype=float)
    obj = qubo_objective(w, Q_mat)
    if obj < best_obj_bf:
        best_obj_bf = obj
        best_bf = bits
t1 = time.time()
selected_bf = [KERNELS_12[i][0] for i,wi in enumerate(best_bf) if wi==1]
print(f"  Sélectionnés: {selected_bf}")
print(f"  Objectif: {best_obj_bf:.4f}")
print(f"  Temps: {(t1-t0)*1000:.1f} ms")

# ── ALGORITHME 2 : Greedy ─────────────────────────────────────────────────
print("
=== Greedy ===")
t0 = time.time()
w_greedy = np.zeros(M)
best_greedy_obj = qubo_objective(w_greedy, Q_mat)
history_greedy = [best_greedy_obj]
for _ in range(M):
    best_m, best_delta = None, np.inf
    for m in range(M):
        if w_greedy[m] == 1: continue
        w_try = w_greedy.copy(); w_try[m] = 1
        obj = qubo_objective(w_try, Q_mat)
        if obj < best_delta:
            best_delta = obj; best_m = m
    if best_m is None or best_delta >= best_greedy_obj:
        break
    w_greedy[best_m] = 1
    best_greedy_obj = best_delta
    history_greedy.append(best_greedy_obj)
t1 = time.time()
selected_greedy = [KERNELS_12[i][0] for i,wi in enumerate(w_greedy) if wi==1]
print(f"  Sélectionnés: {selected_greedy}")
print(f"  Objectif: {best_greedy_obj:.4f}")
print(f"  Temps: {(t1-t0)*1000:.3f} ms")

# ── ALGORITHME 3 : Recuit Simulé ──────────────────────────────────────────
print("
=== Recuit Simulé ===")
t0 = time.time()
w_sa = np.random.randint(0, 2, M).astype(float)
obj_sa = qubo_objective(w_sa, Q_mat)
best_sa, best_obj_sa = w_sa.copy(), obj_sa
T = 2.0; alpha_cool = 0.97; n_iter = 2000
history_sa = [obj_sa]
for iteration in range(n_iter):
    m_flip = np.random.randint(M)
    w_new = w_sa.copy(); w_new[m_flip] = 1 - w_new[m_flip]
    obj_new = qubo_objective(w_new, Q_mat)
    delta = obj_new - obj_sa
    if delta < 0 or np.random.random() < np.exp(-delta/T):
        w_sa, obj_sa = w_new, obj_new
    if obj_sa < best_obj_sa:
        best_obj_sa, best_sa = obj_sa, w_sa.copy()
    T *= alpha_cool
    history_sa.append(best_obj_sa)
t1 = time.time()
selected_sa = [KERNELS_12[i][0] for i,wi in enumerate(best_sa) if wi==1]
print(f"  Sélectionnés: {selected_sa}")
print(f"  Objectif: {best_obj_sa:.4f}")
print(f"  Temps: {(t1-t0)*1000:.1f} ms")
print(f"  Accord avec brute force: {list(best_sa)==list(np.array(best_bf,float))}")

# ── Visualisation comparaison ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.plot(history_sa, '#3498db', lw=1.5, label='Recuit simulé (meilleur)')
ax.axhline(best_obj_bf, color='#e74c3c', ls='--', lw=2, label=f'Brute force={best_obj_bf:.3f}')
ax.axhline(best_greedy_obj, color='#2ecc71', ls=':', lw=2, label=f'Greedy={best_greedy_obj:.3f}')
ax.set_xlabel('Itération'), ax.set_ylabel('Objectif QUBO (min)')
ax.set_title('Convergence du recuit simulé
vs solutions de référence', fontweight='bold')
ax.legend(), ax.grid(alpha=0.3)

ax2 = axes[1]
# Distribuer les poids de chaque méthode
names_k = [k[0] for k in KERNELS_12]
x = np.arange(M)
ax2.bar(x-0.3, best_bf, 0.25, label='Brute Force', color='#e74c3c', alpha=0.8)
ax2.bar(x,     w_greedy, 0.25, label='Greedy', color='#2ecc71', alpha=0.8)
ax2.bar(x+0.3, best_sa,  0.25, label='Recuit simulé', color='#3498db', alpha=0.8)
ax2.set_xticks(x), ax2.set_xticklabels(names_k, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('w_m ∈ {0,1}')
ax2.set_title('Kernels sélectionnés par chaque algorithme
(1=sélectionné, 0=ignoré)', fontweight='bold')
ax2.legend()
plt.tight_layout()
plt.show()
print(f"
Conclusion pour M={M}: Brute Force, Greedy et SA donnent la même solution.")
print("→ Pour M=12, le problème est facile. Le quantique n'est pas nécessaire ici.")


## Section 4 — Le rôle du paramètre λ

### λ = 0 : sélection purement basée sur la performance

$$\min \mathbf{w}^\top \mathbf{Q} \mathbf{w} \xrightarrow{\lambda=0} \min -\sum_m w_m s_m$$

Les kernels avec les $s_m$ les plus élevés sont sélectionnés — **peu importe la redondance**.
Problème : on peut sélectionner 3 kernels de la même famille (très corrélés), qui apportent peu d'information nouvelle.

### λ → ∞ : sélection purement basée sur la diversité

La pénalité de redondance domine → on sélectionne le kernel le plus "isolé" (faible $\sum_{m'} A_{mm'}$), indépendamment de sa performance.
Problème : on peut sélectionner un kernel peu performant.

### λ optimal : compromis performance / diversité

La valeur optimale $\lambda^*$ est celle qui maximise l'AUC réelle du SVM entraîné.
**Comment calibrer :** sweep sur $\lambda \in [0, 3]$ avec 10-20 valeurs, cross-validation pour chaque.

### Interprétation mathématique

Le problème QUBO avec $\lambda > 0$ est un **problème d'optimisation bi-objectif** :
$$\text{Pareto front : max performance vs max diversité}$$

$\lambda$ est le multiplicateur de Lagrange qui paramétrise ce front de Pareto.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from itertools import product as iproduct

np.random.seed(42)

# ── Réutiliser les kernels de la section précédente ──────────────────────
def K_Z(X, alpha=1.0):
    n, d = X.shape
    K = np.ones((n,n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha*diff)**2
    return K

def K_ZZ(X, alpha=1.0):
    K = K_Z(X, alpha)
    n, d = X.shape
    for k in range(d):
        for l in range(k+1, d):
            c = X[:,k]*X[:,l]
            diff = c.reshape(-1,1) - c.reshape(1,-1)
            K *= np.cos(alpha*diff)**2
    return K

def center_kernel(K):
    n = len(K); H = np.eye(n) - np.ones((n,n))/n; return H@K@H

def frobenius_ip(A, B): return np.sum(A*B)

# Données légèrement plus grandes pour mieux mesurer les AUC
rng = np.random.default_rng(42)
n_eval = 40
X_eval = rng.uniform(0, 2, (n_eval, 4))
# Labels: cercle dans l'espace quantique (basé sur la norme)
y_eval = (np.sum((X_eval - 1.0)**2, axis=1) < 1.2).astype(float) * 2 - 1

KERNELS_12 = [
    ('Z α=1.0',  K_Z,  1.0, 0.698),
    ('Z α=3.0',  K_Z,  3.0, 0.712),
    ('ZZ α=1.0', K_ZZ, 1.0, 0.734),
    ('ZZ α=4.0', K_ZZ, 4.0, 0.756),
    ('ZZ α=2.5', K_ZZ, 2.5, 0.738),
    ('ZZ α=0.5', K_ZZ, 0.5, 0.720),
    ('Z α=0.5',  K_Z,  0.5, 0.695),
    ('Z α=2.0',  K_Z,  2.0, 0.705),
    ('ZZ α=1.5', K_ZZ, 1.5, 0.730),
    ('ZZ α=3.0', K_ZZ, 3.0, 0.748),
    ('Z α=1.5',  K_Z,  1.5, 0.701),
    ('ZZ α=2.0', K_ZZ, 2.0, 0.742),
]
M = len(KERNELS_12)
s = np.array([k[3] for k in KERNELS_12])

# Calculer matrices kernel sur les données d'évaluation
Kmats_eval = []
for name, fn, alpha, _ in KERNELS_12:
    K = fn(X_eval, alpha)
    np.fill_diagonal(K, 1.0)
    Kmats_eval.append(K)

# Alignements mutuels
Kmats_c = [center_kernel(K) for K in Kmats_eval]
A_mat = np.zeros((M, M))
for m in range(M):
    for mp in range(M):
        num = frobenius_ip(Kmats_c[m], Kmats_c[mp])
        den = np.sqrt(frobenius_ip(Kmats_c[m],Kmats_c[m]) * frobenius_ip(Kmats_c[mp],Kmats_c[mp]))
        A_mat[m,mp] = num/den if den>0 else 0.

def solve_qubo_bf(s, A_mat, lam):
    '''Résoudre QUBO par brute force.'''
    M = len(s)
    Q = lam * A_mat.copy()
    np.fill_diagonal(Q, -s)
    best_w, best_obj = None, np.inf
    for bits in iproduct([0,1], repeat=M):
        w = np.array(bits, dtype=float)
        obj = w @ Q @ w
        if obj < best_obj:
            best_obj, best_w = obj, bits
    return np.array(best_w, dtype=float)

def eval_selection(w, Kmats, y):
    '''AUC du kernel combiné par la sélection w.'''
    selected = [K for K, wi in zip(Kmats, w) if wi == 1]
    if not selected: return 0.5
    K_comb = np.mean(selected, axis=0)
    eigs = np.linalg.eigvalsh(K_comb)
    if eigs.min() < 0: K_comb += (-eigs.min()+1e-8)*np.eye(len(K_comb))
    try:
        return cross_val_score(SVC(kernel='precomputed',C=1.), K_comb, y,
                               cv=4, scoring='roc_auc').mean()
    except: return 0.5

# ── Sweep sur λ ───────────────────────────────────────────────────────────
lambdas = np.linspace(0, 3, 20)
aucs_lambda = []
n_selected_lambda = []
selected_names_lambda = []

for lam in lambdas:
    w_opt = solve_qubo_bf(s, A_mat, lam)
    auc = eval_selection(w_opt, Kmats_eval, y_eval)
    n_sel = int(w_opt.sum())
    sel = [KERNELS_12[i][0] for i,wi in enumerate(w_opt) if wi==1]
    aucs_lambda.append(auc)
    n_selected_lambda.append(n_sel)
    selected_names_lambda.append(sel)

# Trouver le λ optimal
best_lam_idx = np.argmax(aucs_lambda)
best_lam = lambdas[best_lam_idx]
print(f"λ optimal: {best_lam:.2f}")
print(f"AUC à λ*: {aucs_lambda[best_lam_idx]:.4f}")
print(f"Kernels sélectionnés: {selected_names_lambda[best_lam_idx]}")

# ── Graphique double axe ──────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

line1, = ax1.plot(lambdas, aucs_lambda, '#3498db', lw=2.5, marker='o', ms=6, label='AUC')
line2, = ax2.plot(lambdas, n_selected_lambda, '#e74c3c', lw=2.5, marker='s', ms=6,
                   ls='--', label='Nb kernels sélectionnés')
ax1.axvline(best_lam, color='gold', lw=2, ls='-', label=f'λ* = {best_lam:.2f}')
ax1.scatter([best_lam], [aucs_lambda[best_lam_idx]], s=150, color='gold', zorder=5)

ax1.set_xlabel('Paramètre λ', fontsize=11)
ax1.set_ylabel('AUC (4-fold cross-val)', fontsize=11, color='#3498db')
ax2.set_ylabel('Nombre de kernels sélectionnés', fontsize=11, color='#e74c3c')
ax1.tick_params(axis='y', labelcolor='#3498db')
ax2.tick_params(axis='y', labelcolor='#e74c3c')
ax2.set_yticks(range(0, M+1, 2))

# Annoter les zones
ax1.axvspan(0, 0.3, alpha=0.1, color='orange', label='λ trop petit: redondant')
ax1.axvspan(2.0, 3.0, alpha=0.1, color='red', label='λ trop grand: 1 kernel')
ax1.axvspan(0.3, 2.0, alpha=0.1, color='green', label='Zone optimale')

lines = [line1, line2, plt.Line2D([],[],color='gold',lw=2,label=f'λ*={best_lam:.2f}')]
ax1.legend(handles=lines + [
    plt.mpatches.Patch(color='orange', alpha=0.4, label='Trop redondant'),
    plt.mpatches.Patch(color='red', alpha=0.4, label='1 seul kernel'),
    plt.mpatches.Patch(color='green', alpha=0.4, label='Zone optimale'),
], fontsize=8, loc='lower right')

ax1.set_title('Sweep sur λ : AUC et nombre de kernels sélectionnés
'
              f'Optimum à λ*={best_lam:.2f} → {selected_names_lambda[best_lam_idx]}',
              fontweight='bold')
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Section 5 — Connexion au recuit quantique : quand M devient grand

### Formulation Ising

Le problème QUBO peut être converti en problème d'Ising (spins $\pm 1$) via le changement de variable $w_m = (1 - \sigma_m)/2$ :
$$H_{\text{Ising}} = -\sum_m J_m \sigma_m - \sum_{m < m'} J_{mm'} \sigma_m \sigma_{m'}$$

C'est le hamiltonien natif des annealers quantiques (D-Wave, Pasqal) et la cible des algorithmes QAOA sur IBM.

### Recuit quantique : principe

Un annealer quantique commence dans un état de superposition quantique de toutes les solutions ($2^M$ états en parallèle), puis "refroidit" progressivement en évoluant vers l'état d'énergie minimale.

**Avantage potentiel :** le tunneling quantique permet d'explorer les barrières d'énergie plus efficacement que le recuit thermique classique pour certains paysages.

**Limitation importante :** C'est une **heuristique** — aucune garantie de trouver l'optimum global. La complexité empirique est $\approx O(M^{1.5})$ sur les benchmarks publics, mais ce n'est pas prouvé.

### Break-even : quand le quantique vaut-il le coût ?

| $M$ | Brute force | Greedy classique | Recuit quantique (empirique) |
|-----|-------------|-----------------|------------------------------|
| 12  | 4 096 = **instantané** | **instantané** | Coût d'accès cloud > bénéfice |
| 25  | 33M = 330ms | **ms** | Comparable au greedy |
| 50  | $10^{15}$ = **années** | minutes | **Avantage potentiel** |
| 100 | Impossible | heures | Potentiellement utile |

**Conclusion honnête pour notre projet :** À $M=12$, le quantique n'est pas nécessaire pour QUBO. La valeur est dans la **formulation** (binary optimization framework), pas dans l'exécution quantique.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Courbes de complexité ────────────────────────────────────────────────
M_vals = np.array([5, 8, 10, 12, 15, 20, 25, 30, 35, 40, 50])

# Normalisés pour que M=12 soit le point de référence (1.0 unité)
M_ref = 12
classical_bf   = 2.**M_vals / (1e8)                   # secondes, @100M éval/s
heuristic      = M_vals**3   / (1e8 * M_ref**3 / 4096) # relatif
quantum_anneal = M_vals**1.5 / (1e8 * M_ref**1.5 / 100) # relatif, empirique

fig, ax = plt.subplots(figsize=(12, 6))

# Zones colorées
ax.axvspan(M_vals[0]-1, 25,   alpha=0.12, color='#27ae60')
ax.axvspan(25,          35,   alpha=0.12, color='#f39c12')
ax.axvspan(35, M_vals[-1]+2,  alpha=0.12, color='#3498db')

ax.semilogy(M_vals, classical_bf,   'r--o', lw=2.5, ms=7, label='Brute Force O(2^M) — exact')
ax.semilogy(M_vals, heuristic,      color='#e67e22', ls=':', marker='s', ms=7, lw=2.5,
            label='Heuristique classique O(M³)')
ax.semilogy(M_vals, quantum_anneal, 'b-^', lw=2.5, ms=7,
            label='Recuit quantique O(M^1.5) ★ empirique')

# Annotations
for m_ann, label, color, xytext in [
    (12, 'Notre projet
M=12
4 096 ss-ensembles', '#27ae60', (14, 1e2)),
    (25, 'Limite classique
33 millions', '#d35400',          (27, 1e4)),
    (50, 'M=50:
~10¹⁵
Quantique seul
viable',  '#2980b9',  (42, 1e9)),
]:
    idx = np.where(M_vals == m_ann)[0]
    if len(idx) > 0:
        ax.annotate(label, xy=(m_ann, classical_bf[idx[0]]), xytext=xytext,
                    fontsize=8.5, color=color, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5),
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9))

ax.axvline(12, color='#27ae60', lw=1.5, ls='--', alpha=0.7)
ax.axvline(25, color='#d35400', lw=1.5, ls='--', alpha=0.7)

# Labels zones
ax.text(17, 1e-5, 'Zone classique
(M < 25)', ha='center', color='#1a7a3c', fontsize=9, fontweight='bold')
ax.text(30, 1e-5, 'Transition', ha='center', color='#b7770d', fontsize=9, fontweight='bold')
ax.text(43, 1e-5, 'Zone quantique
(M > 35)', ha='center', color='#1a5276', fontsize=9, fontweight='bold')

ax.set_xlabel('Nombre de kernels M', fontsize=11)
ax.set_ylabel('Temps de calcul (secondes, brute force)', fontsize=11)
ax.set_title('Explosion combinatoire 2^M : quand le quantique devient-il nécessaire ?
'
             '★ = heuristique empirique, pas de garantie théorique', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.25)
ax.set_xlim(M_vals[0]-1, M_vals[-1]+2)

# Axe temporel
y_ticks = [1e-3, 1e0, 1e3, 1e6, 1e9, 1e12]
y_labels = ['1 ms', '1 s', '17 min', '11 jours', '30 ans', '31 000 ans']
ax2 = ax.twinx()
ax2.set_yscale('log')
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks(y_ticks)
ax2.set_yticklabels(y_labels, fontsize=8, color='#7f8c8d')
ax2.set_ylabel('Durée estimée', fontsize=9, color='#7f8c8d')

plt.tight_layout()
plt.show()

# Tableau récapitulatif
print("Tableau de décision : QUBO classique vs quantique")
print(f"{'M':5s} | {'2^M':12s} | {'BF (s)':10s} | {'Greedy (ms)':12s} | Recommandation")
print("-"*65)
for m in [12, 20, 25, 35, 50]:
    bf_s = 2**m / 1e8
    gr_ms = m**3 / 1e5
    if m <= 12:
        rec = "Brute force OK"
    elif m <= 25:
        rec = "Greedy/SA classique"
    elif m <= 35:
        rec = "SA classique ou QUBO quantique"
    else:
        rec = "Quantique préféré (D-Wave / QAOA)"
    print(f"{m:5d} | {2**m:12,.0f} | {bf_s:10.2e} | {gr_ms:12.1f} | {rec}")
